<a href="https://colab.research.google.com/github/robinlwong/5m-data-2.8-out-of-core-computation/blob/main/assignment/5m_data_2_8_polars_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Question 1

Question: Join the `metadata_pl` and `ratings_pl` DataFrames in Polars, then calculate the average rating by `genre` and by `original_language` (separately).

#### **The Lazy Polars Pipeline**

In [ ]:
import polars as pl
"""
If data !loaded into mem use `pl.scan_csv("movies_metadata.csv")`
instead of `pl.read_csv()` so the data never fully loads into RAM at all.
1. Lazy Query Plan.
"""
joined_lazy = (
    metadata_pl.lazy()      # Convert to LazyFrame.
# Convert to Int64, take the "id" column which was text/strings
# and casts it to 64-bit integers to match the data tye in ratings_pl
# strict=false prevents a crash if !number, return null.
    .with_columns(pl.col("id").cast(pl.Int64, strict=False))
    .drop_nulls(subset=["id"])  # Drop nulls because you can't join on null ID
# Ensure unique "id" to prevent memory explosion as duplicate movie IDs
# would multiply against millions of rating rows which would result in an
# an explosion of many-to-many joins which would freeze the server.
    .unique(subset=["id"])
    .join(
        ratings_pl.lazy(),  # Convert to LazyFrame
        left_on="id",
        right_on="movieId",
        how="inner"         # Only keep rows where movie exists in both tables
    )
)

# ---------- Aggregate and Execute ----------
# 2. Calculate avg_rating by genre using joined_lazy from 1.
avg_rating_by_genre = (
    joined_lazy
    .group_by("genre")
    .agg(pl.col("rating").mean().alias("avg_rating"))
    .collect()    # Execute optimized plan on CPU.
)


# 3. Calculate avg_rating by original_language, use joined_lazy from 1.
avg_rating_by_language = (
    joined_lazy
    .group_by("original_language")
    .agg(pl.col("rating").mean().alias("avg_rating"))
    .collect()
)

# 4. Result.
print(avg_rating_by_genre)
print(avg_rating_by_language)

### Question 2

Question: Calculate the average total amount for vendors with at least 5 trips from the NYC Taxi Trip Data.

In [ ]:
import polars as pl

"""
1. Lazy query plan.
Read data in chunks, automatically convert text dates
into datetime objects just in case math needs to be done.
"""
q = (
    pl.scan_csv('data/taxi_trip_data.csv', try_parse_dates=True)
# Calculate metrics for each specific vendor,
# tells Polars to organize all the millions of rows into distinct
# buckets based on the unique 'vendor_id'.
    .group_by("vendor_id")
    .agg(
        pl.mean("total_amount").alias("avg_total_amount"),
        pl.len().alias("trip_count")
    )
    .filter(pl.col("trip_count") >= 5)
)

# Execute.
df_pl = q.collect()
print(df_pl)